# 00. Battery PdM system — overview

A predictive maintenance system for telecom backup batteries.

**What this notebook covers:** the goal, the architecture at a glance, the headline
results, and where to look for deeper dives. Read this first.


## The problem

Telecom base stations have backup batteries that kick in during grid outages.
In Pakistan, daily load-shedding means these batteries discharge and recharge
constantly — they wear out far faster than spec sheets predict.

A NOC operator needs answers to two questions, on two different cadences:

| Question | Cadence | Action |
|----------|---------|--------|
| **Will this site drain in the *next 48 hours*?** | Daily | Proactive generator scheduling |
| **Should this battery be *replaced*?** | Weekly | Add to next month's replacement run |

We build two models sharing the same alarms-only feature pipeline. (A third
model — real-time autonomy estimation — was decommissioned after analysis
showed the drain predictor subsumes it. See notebook 04 for the comparison.)

## The data — alarms only, no telemetry

Real telecom NOCs usually don't have reliable voltage/temperature telemetry from
sites — what they have is the **alarm stream** from the Network Management System
(NMS): events like `AC_MAINS_FAIL`, `RECTIFIER_FAULT`, `LOAD_DISCONNECT`.

We deliberately constrain ourselves to alarms + static site config (load, capacity,
region) + the regional load-shedding schedule. No telemetry features.

This makes the problem harder *and* more realistic — and the lift over baselines
ends up being more honest.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUTS = Path("..") / "outputs"
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

alarms = pd.read_parquet(OUTPUTS / "alarms.parquet")
sites = pd.read_parquet(OUTPUTS / "site_static.parquet")

print(f"Alarms: {len(alarms):,}    Sites: {len(sites):,}    "
      f"Regions: {sites['region'].nunique()}")
print(f"Alarm time range: {alarms['timestamp_h'].min():.1f}h to "
      f"{alarms['timestamp_h'].max():.1f}h "
      f"({(alarms['timestamp_h'].max() - alarms['timestamp_h'].min())/(24*30):.1f} months)")


Alarms: 203,043    Sites: 500    Regions: 5
Alarm time range: 0.0h to 25919.0h (36.0 months)


In [2]:
# Regional breakdown — what we're predicting on
merged = alarms.merge(sites[["site_id", "region"]], on="site_id")
codes = ["AC_MAINS_FAIL", "LOAD_DISCONNECT", "RECTIFIER_FAULT", "BATT_UNDERVOLTAGE"]
by_region = (merged.groupby(["region", "alarm_code"]).size()
             .unstack(fill_value=0)[codes])
print("Alarm counts per region:")
by_region


Alarm counts per region:


alarm_code,AC_MAINS_FAIL,LOAD_DISCONNECT,RECTIFIER_FAULT,BATT_UNDERVOLTAGE
region,,,,
islamabad,7877,28,1110,593
karachi,18183,2715,1097,7606
lahore,24895,12779,1081,17552
peshawar,24977,15511,1075,19474
quetta,19850,4849,1055,10416


**Key finding from the data itself:** drain rate per outage varies massively by region.

| Region | Drain per outage |
|--------|-----------------:|
| Islamabad | **0.1%** (almost never) |
| Karachi | 6.2% |
| Quetta | 10.4% |
| Lahore | **24.9%** |
| Peshawar | **29.7%** |

A single global model is asked to fit 5 fundamentally different physics. The
per-region experiment in notebook 02 shows why this still works (region is
already a feature) and when it wouldn't.


## The two models — headline results

| Model | Algorithm | Metric | Score |
|-------|-----------|--------|------:|
| Drain in next 48h | XGBoost binary:logistic | AUC | **0.88** |
| Long-term failure | XGBoost survival:cox | C-index | **0.90** |

The drain predictor's per-region AUC ranges from 0.76 (Peshawar) to 0.87 (Islamabad).
The variation matters more than the average — it's a signal that **calibration**
and **per-region monitoring** are the actual operational priorities, not raw AUC.

Isotonic calibration reduces Brier score by ~42% (0.128 to 0.074), meaning the
predicted probabilities closely match observed drain rates.

The autonomy model (hours-to-LVD, C-index 0.73) was decommissioned — see notebook 04
for the analysis showing the drain predictor subsumes its utility.

## The system around the models

What makes this production-grade (not just a Jupyter notebook):

**ML Lifecycle (Santiago pattern):**
- **10 Metaflow flows** orchestrating scoring, drift detection, retraining,
  shadow promotion, rollback, and deployment
- **Proactive weekly retraining** — always trains a challenger, CV-gated promotion.
  Drift monitor is a safety net, not the trigger.
- **Champion/challenger comparison** on the same held-out test set (apples to apples)
- **Shadow deployment** — challenger scores in parallel with champion, promoted
  only when realized production labels prove improvement (blue/green for batch ML)
- **Automated rollback** — if promoted model's Brier degrades >10% within 48h,
  auto-reverts to archived champion
- **PSI-based drift detection** with training-time reference profile

**ML Engineering:**
- **Feature hash validation** between training and inference (catches train/serve skew)
- **Isotonic probability calibration** on held-out calibration set (Brier -42%)
- **Cost-aware threshold optimization** with sensitivity analysis
- **Cold-start fallback** — sites with no history get regional prior
- **Model version history** — each run persists to `v_<timestamp>/`; never overwritten

**AWS Infrastructure:**
- **Step Functions** orchestrating Metaflow flows as Batch jobs
- **Fargate Spot** — $0 idle, pay per-second only when jobs run
- **EventBridge** — 7 scheduled cron rules (daily scoring, weekly retraining)
- **Terraform IaC** — full infrastructure as code, portable across accounts
- **GitHub Actions CI/CD** — lint + test + build + deploy + register state machines
- **NOC dashboard** on ECS (Streamlit) with ALB

See `SETUP.md` for the full deployment guide.

## Where to look next

- [`01_drift_detection_demo.ipynb`](01_drift_detection_demo.ipynb) — the most
  compelling narrative: a regional grid upgrade silently breaks the model,
  drift monitoring catches it, and we explain *why retraining alone doesn't fix it*
- [`02_model_evaluation_per_region.ipynb`](02_model_evaluation_per_region.ipynb)
  — calibration analysis, per-region AUC, the experiment that disproved per-region
  modeling
- [`03_hyperparameter_tuning.ipynb`](03_hyperparameter_tuning.ipynb) — Optuna
  HPO vs defaults (honest finding: +0.5-1% lift, not worth the compute)
- [`04_autonomy_vs_drain_comparison.ipynb`](04_autonomy_vs_drain_comparison.ipynb)
  — historical analysis that led to decommissioning the autonomy model
- `src/battery_pdm/common/features.py` — the feature pipeline shared by both
  models (vectorized, deterministic encoding)
- `src/battery_pdm/monitoring/drift.py` — the PSI math
- `src/battery_pdm/monitoring/model_registry.py` — versioned artifacts, isotonic
  calibration, atomic trigger handling
- `src/battery_pdm/monitoring/threshold.py` — cost-aware dispatch optimization
- `src/battery_pdm/flows/retraining_flow.py` — champion/challenger with CV gate
- `src/battery_pdm/flows/shadow_promotion_flow.py` — blue/green for batch ML
- `SETUP.md` — full deployment guide (local + AWS in 30 minutes)